# GAN 추가 학습: 부분 가리기 기준 버전
**이 노트북은 설정한 실험 하나만 실행합니다. 다른 버전을 자동 실행하지 않습니다.**
출발점은 외부 사전학습 생성기가 아닌, 우리가 LoveDA로 직접 학습한 `semantic_04000`입니다.
원본 지도/원본 RGB로 학습하고, 생성 단계에서 수정 지도를 넣습니다. 전체 가리기는 선택한 패치의 RGB 전체만 가리며 지도와 정답은 유지합니다.
새 방법의 개선 효과는 미검증입니다. 기본256 native crop이며512는 메모리 확인 후 별도 실험으로 사용하세요.
전체학습 순서: 환경 확인 → 설정 → Train 데이터 → 가림계획 확인 → 작은 점검 → 학습 → 생성·평가.
별도 셀로 평가만 실행할 수 있습니다. GPU 서버의 CUDA PyTorch는 담당자가 맞춰 설치해야 합니다.

In [ ]:
import os, sys, json, subprocess, time, zipfile, signal
from pathlib import Path
candidates = [Path.cwd(), Path.cwd() / '05_서버학습코드', Path.cwd() / '1022_진행상황' / '05_서버학습코드']
ROOT = next((p.resolve() for p in candidates if (p / 'train.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('노트북과 train.py가 있는 폴더에서 실행하거나 ROOT를 그 폴더로 지정하세요.')
print('package:', ROOT, '\npython:', sys.executable)
# torch/torchvision은 CUDA에 맞는 버전을 먼저 설치합니다.
# 필요한 보조 패키지만 설치합니다. 기존 torch를 자동으로 교체하지 않습니다.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)
import torch, torchvision
if tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2]) < (2, 4):
    raise RuntimeError('torch2.4 이상과 호환되는 torchvision이 필요합니다.')
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__, 'CUDA:', torch.cuda.is_available())
print('설치 안내: https://pytorch.org/get-started/locally/')

## 설정
`NUM_TRAIN_IMAGES=None`이면 평가 원본32장을 제외한2,490장 전체를 사용합니다.
`PATCHES_PER_IMAGE`는 한 계획 주기에서 원본당 만드는 후보 crop 수입니다. 균형 선택 때문에 모든 후보가 같은 횟수로 쓰이지는 않습니다. 다음 주기에는 새 위치를 만듭니다.
`BALANCE_STRENGTH=0`은 후보 균등 선택, `1`은 사용 가능한 클래스의 평균 가림 면적 비율을 균등 쪽으로 유도합니다. 정확한 균등을 보장하지 않습니다.
`SEED=None`이면 새 시드를 뽑아 config에 기록합니다. 새 실험은 RUN_NAME을 바꾸세요. 같은 이름의 기존 실험에서는 기록된 시드를 재사용합니다.
`TRAIN=False`로 두면 학습 없이 지정한 체크포인트를 생성·평가할 수 있습니다.

In [ ]:
RUN_NAME = 'partial_only'
NUM_TRAIN_IMAGES = 256   # None: 전체2,490장. 2..2490 선택 가능.
PATCH_SIZE = 256         # 256 또는512(native crop);512는 GPU 점검 후 사용
PATCHES_PER_IMAGE = 4
FULL_IMAGE_RATIO = 0.0  # 나머지는 부분 가리기. 객체 전체 가리기는 이번 버전에서 제외.
BALANCE_STRENGTH = 0.3
SEED = 42               # None이면 새 시드 자동 선택 및 기록
TOTAL_UPDATES = 4000     # 학습 반복 예산; AMP 실제 G/D 업데이트 수는 별도 기록
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
AMP = True
SAVE_EVERY = 100
GPU_INDEX = '0'
DEVICE = 'auto'          # GPU 없으면 본학습 중단. CPU는 작은 점검에만 권장
INITIAL_CHECKPOINT = 'weights/semantic_04000.pt'
DATA_ROOT = ROOT / 'external_data' / 'LoveDA'  # 기존 Train/Urban,Train/Rural의 상위 경로로 변경 가능
TRAIN_ZIP = None         # 이미 받은 Train.zip 경로 또는 None
DOWNLOAD_TRAIN_IF_NEEDED = True  # 첫 다운로드 약4GB, 압축해제/결과까지 여유20GB 권장
TRAIN = True
RESUME = True
RUN_SMOKE_CHECK = True   # 임시 모델의 실제3회 점검; 본실험/원래 가중치는 변경하지 않음
EVALUATION_CHECKPOINT = None  # None이면 이 실험 latest; 평가만 할 때 원하는 경로 지정
RUN_AUDIT = True
RUN_GLOBAL_GENERATION_PROBE = False
if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ['.', '..']:
    raise ValueError('RUN_NAME은 폴더 이름 하나로 지정하세요.')
RUN_DIR = ROOT / 'runs' / RUN_NAME
print('run folder:', RUN_DIR)

In [ ]:
CHILD_ENV = os.environ.copy()
CHILD_ENV['CUDA_VISIBLE_DEVICES'] = GPU_INDEX
def run_script(script, *arguments, stop_file=None):
    command = [sys.executable, '-u', str(ROOT / script), *map(str, arguments)]
    process = subprocess.Popen(command, cwd=ROOT, env=CHILD_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    try:
        for line in process.stdout:
            print(line, end='')
        result = process.wait()
    except KeyboardInterrupt:
        if stop_file is not None:
            Path(stop_file).write_text('notebook interrupt requested', encoding='utf-8')
            print('현재 배치 완료 후 저장하도록 STOP을 요청했습니다. 프로세스가 끝날 때까지 기다리세요.')
            process.wait()
        else:
            process.send_signal(signal.SIGINT)
            process.wait()
        raise
    if result:
        raise RuntimeError(f'{script} exited with status {result}; inspect the printed error.')
preflight_args = ['--out', ROOT / 'checks' / 'environment', '--device', DEVICE]
if TRAIN and DEVICE != 'cpu': preflight_args += ['--require-cuda']
run_script('preflight.py', *preflight_args)

## 데이터와 학습 계획
기존 데이터가 있으면 DATA_ROOT만 맞추면 됩니다. 없으면 공식 Train ZIP을 받고 기존 평가 원본을 제외해 추출합니다.
학습 전 입력 미리보기와 클래스 분포를 확인하세요. 전체 픽셀 균등이 아닌, 모드별 가림 면적 비율의 완만한 보정입니다.
설정·시드·코드·데이터가 바뀐 채 기존 체크포인트를 이어 쓰면 오류를 냅니다. 새 실험은 새 RUN_NAME으로 실행하세요.

In [ ]:
CONFIG_PATH = ROOT / 'run_configs' / (RUN_NAME + '.json')
CONFIG_PATH.parent.mkdir(exist_ok=True)
config = dict(num_train_images=NUM_TRAIN_IMAGES, patch_size=PATCH_SIZE, patches_per_image=PATCHES_PER_IMAGE,
              updates=TOTAL_UPDATES, batch_size=BATCH_SIZE, full_image_ratio=FULL_IMAGE_RATIO,
              balance_strength=BALANCE_STRENGTH, seed=SEED, lr=LEARNING_RATE,
              save_every=SAVE_EVERY, amp=AMP, cpu_threads=4, initial_checkpoint=str(INITIAL_CHECKPOINT))
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')
if TRAIN:
    args = ['--data-root', DATA_ROOT]
    if TRAIN_ZIP is not None: args += ['--train-zip', TRAIN_ZIP]
    if DOWNLOAD_TRAIN_IF_NEEDED: args += ['--download']
    run_script('prepare_data.py', *args)
    run_script('train.py', '--config', CONFIG_PATH, '--data-root', DATA_ROOT, '--out', RUN_DIR, '--prepare-only')
    run_script('preflight.py', '--out', RUN_DIR / 'checks', '--run', RUN_DIR, '--data-root', DATA_ROOT, '--device', DEVICE)
    from IPython.display import display, Image as DisplayImage
    display(DisplayImage(filename=str(RUN_DIR / 'checks' / 'training_input_preview.png')))
    distribution = json.loads((RUN_DIR / 'class_distribution.json').read_text())
    print('순서: 배경, 건물, 도로, 물, 나지, 숲, 농경지')
    print(json.dumps(distribution, ensure_ascii=False, indent=2))
    print('계획:', TOTAL_UPDATES * BATCH_SIZE, 'draws; 시드:', json.loads((RUN_DIR/'config.json').read_text())['resolved_seed'])

## 실제 데이터로 짧은 실행 점검
원래 가중치는 읽기만 하고, checks 폴더의 임시 모델로 부분/전체 가리기와 저장·재개를 검사합니다.
이 점검 통과는 생성 품질 개선을 의미하지 않습니다. GPU메모리가 부족하면 BATCH_SIZE를 줄여 새 이름으로 실험하세요.

In [ ]:
if TRAIN and RUN_SMOKE_CHECK:
    stamp = time.strftime('%Y%m%d_%H%M%S')
    run_script('smoke_check.py', '--data-root', DATA_ROOT, '--out', ROOT / 'checks' / ('smoke_' + stamp), '--device', DEVICE)

## 본학습 - 현재 설정 하나만 실행
중단하려면 RUN_DIR에 STOP이라는 파일을 만들거나 이 셀을 중단하세요. 다음 배치 경계에서 저장합니다.
강제 커널/서버 종료는 마지막 정기 저장 이후 진행을 잃을 수 있습니다. 재개 시 STOP 파일은 확인 후 직접 제거하세요.
실행 중 코드·학습 데이터·설정을 변경하지 마세요. 완료된 실험을 자동 재학습하지 않습니다.

In [ ]:
if TRAIN:
    args = ['--config', CONFIG_PATH, '--data-root', DATA_ROOT, '--out', RUN_DIR, '--device', DEVICE]
    if RESUME: args += ['--resume']
    run_script('train.py', *args, stop_file=RUN_DIR / 'STOP')
    print(json.loads((RUN_DIR / 'status.json').read_text()))

## 저장 모델로 생성·평가
35개 과거 개발 사례 +16개 Val 개발 사례를 사용합니다. 모두 개발 자료이며 최종 테스트가 아닙니다.
원래/수정/제거 지도 출력을 비교합니다. 지도 제거는 학습되지 않은 입력일 수 있어 보조 진단입니다.
분할기 픽셀 인식 비율은 편집 성공률이 아닙니다. 실제 원본에서의 평가기 오류도 같이 보고 human_review.csv에 형태·경계·질감 판정을 남기세요.
GAN은 고정 입력에서 결정적입니다. 생성 seed를 바꿔 다양한 사진이 나오는 모델이 아닙니다.

In [ ]:
checkpoint = Path(EVALUATION_CHECKPOINT) if EVALUATION_CHECKPOINT else RUN_DIR / 'checkpoints' / 'latest.pt'
if not checkpoint.is_absolute(): checkpoint = ROOT / checkpoint
evaluation_args = ['--checkpoint', checkpoint, '--out', RUN_DIR / 'evaluation', '--device', DEVICE]
if not RUN_AUDIT: evaluation_args += ['--no-audit']
if RUN_GLOBAL_GENERATION_PROBE: evaluation_args += ['--global-probe']
run_script('evaluate.py', *evaluation_args)
print('비교 그림:', RUN_DIR / 'evaluation' / 'index.html')
print('사람 검토표:', RUN_DIR / 'evaluation' / 'human_review.csv')

## 선택: 기존 기준 모델 또는 다른 실험과 비교
아래 셀은 저장 결과만 비교하며 다른 모델을 학습하지 않습니다. 필요할 때만 실행하세요.

In [ ]:
COMPARE_WITH_BASELINE = True
OTHER_EVALUATION_DIRS = []  # 예: ROOT/'runs'/'another_run'/'evaluation'
folders = [RUN_DIR / 'evaluation']
if COMPARE_WITH_BASELINE:
    baseline_dir = ROOT / ('baseline_evaluation_audit' if RUN_AUDIT else 'baseline_evaluation_noaudit')
    args = ['--checkpoint', ROOT / 'weights' / 'semantic_04000.pt', '--out', baseline_dir, '--device', DEVICE]
    if not RUN_AUDIT: args += ['--no-audit']
    run_script('evaluate.py', *args)
    folders.append(baseline_dir)
folders += [Path(p) for p in OTHER_EVALUATION_DIRS]
if len(folders) > 1:
    run_script('compare_runs.py', '--evaluations', *folders, '--out', RUN_DIR / 'comparison_with_saved_runs')

## 선택: 미사용 Val 최종 테스트 - 기본 꺼짐
개발 결과로 모델/설정을 정한 뒤에만 켜세요. 공식 Val 데이터 또는 Val.zip을 별도로 준비해야 합니다.
예약된1,107개 원본 풀에서 라벨 기하만으로 사례를 고정하며, 모델 출력을 보고 좋은 사례를 고르지 않습니다.
테스트를 보고 모델을 다시 선택하면 그 결과는 개발 평가로 취급해야 합니다.

In [ ]:
RUN_FINAL_TEST = False
VAL_ROOT = None  # Val/Urban,Val/Rural의 상위 데이터 경로 또는 None
VAL_ZIP = None   # 공식 Val.zip 경로 또는 None
NUM_TEST_CASES = 28
TEST_SELECTION_SEED = 1729
TEST_CASES_DIR = ROOT / 'reserved_test_cases'
if RUN_FINAL_TEST:
    args = ['--out', TEST_CASES_DIR, '--count', NUM_TEST_CASES, '--seed', TEST_SELECTION_SEED]
    if VAL_ROOT is not None: args += ['--val-root', VAL_ROOT]
    if VAL_ZIP is not None: args += ['--val-zip', VAL_ZIP]
    run_script('make_test_cases.py', *args)
    args = ['--checkpoint', checkpoint, '--out', RUN_DIR / 'final_test', '--device', DEVICE,
            '--cases-dir', TEST_CASES_DIR, '--final-test']
    if not RUN_AUDIT: args += ['--no-audit']
    run_script('evaluate.py', *args)

## 결과 전달용 ZIP
서버에는 전체 체크포인트를 남기고 최신 체크포인트·계획·설정·평가를 묶습니다. 이전 체크포인트와 중간 스냅샷은 제외합니다.
human_review.csv를 작성한 다음 ZIP을 다시 만들면 그 판정도 포함됩니다.

In [ ]:
EXPORT_RESULTS = True
if EXPORT_RESULTS and RUN_DIR.exists():
    archive_path = ROOT / (RUN_NAME + '_results.zip')
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for path in sorted(RUN_DIR.rglob('*')):
            if not path.is_file() or path.name.endswith('.part') or path.name.endswith('.lock'): continue
            if path.parent.name == 'checkpoints' and path.name not in ['latest.pt','latest.json']: continue
            z.write(path, RUN_NAME + '/' + path.relative_to(RUN_DIR).as_posix())
    print('전달 파일:', archive_path, 'bytes:', archive_path.stat().st_size)